<a href="https://colab.research.google.com/github/marcocslima/dev/blob/master/XML_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#SETUP

In [ ]:
!pip install groq pandas openpyxl
!pip install chardet -q

In [2]:
from google.colab import userdata

# Recupera a chave de API dos "Segredos" do Colab
api_key = userdata.get('GROQ_API_KEY')

In [3]:
# ============================================================
# AGENTE XML → DATAFRAME  |  Multi-Schema  |  Powered by Groq
# ============================================================
# !pip install groq pandas openpyxl

import os
import re
import json
import glob
import hashlib
import chardet
import xml.etree.ElementTree as ET
import pandas as pd
from groq import Groq

# ── CONFIG ───────────────────────────────────────────────────
GROQ_API_KEY = api_key #os.environ.get("GROQ_API_KEY", "sua_chave_aqui")
MODELO       = "llama-3.3-70b-versatile"   # ou "openai/gpt-oss-120b"

# Similaridade mínima de Jaccard para reutilizar um esquema já mapeado
LIMIAR_SIMILARIDADE = 0.90

# ── HELPERS XML ──────────────────────────────────────────────

def _strip_ns(tag):
    return tag.split("}", 1)[1] if "}" in tag else tag

def _detect_encoding(xml_path):
    """Detecta o encoding real do arquivo lendo os bytes brutos."""
    with open(xml_path, 'rb') as f:
        raw = f.read()
    # Primeiro tenta pegar do cabeçalho XML
    import re
    match = re.search(rb'encoding=["\']([^"\']+)["\']', raw[:200])
    if match:
        return match.group(1).decode('ascii')
    # Fallback: chardet
    detected = chardet.detect(raw)
    return detected.get('encoding', 'utf-8')

def _load_xml(source):
    """Carrega XML de caminho ou string, respeitando o encoding correto."""
    if isinstance(source, str) and os.path.isfile(source):
        encoding = _detect_encoding(source)
        with open(source, 'r', encoding=encoding, errors='replace') as f:
            content = f.read()
        # Remove o BOM se existir
        content = content.lstrip('\ufeff')
        return ET.fromstring(content.encode('utf-8'),
                             parser=ET.XMLParser(encoding='utf-8'))
    elif isinstance(source, str):
        return ET.fromstring(source.encode('utf-8'),
                             parser=ET.XMLParser(encoding='utf-8'))
    elif isinstance(source, bytes):
        return ET.fromstring(source)
    else:
        raise ValueError("source deve ser caminho de arquivo ou string XML")

def _collect_extractable_fields(element, prefix="", fields=None):
    if fields is None:
        fields = set()
    tag = _strip_ns(element.tag)
    current_path = f"{prefix}.{tag}" if prefix else tag
    children = [c for c in element if isinstance(c.tag, str)]
    for attr in element.attrib:
        fields.add(f"{current_path}[@{_strip_ns(attr)}]")
    if not children:
        fields.add(current_path)
    for child in children:
        _collect_extractable_fields(child, current_path, fields)
    return fields

def _parse_path(path):
    match = re.match(r"^(.*)\[@([^\]]+)\]$", path)
    if match:
        return match.group(1).split("."), match.group(2)
    return path.split("."), None

def _get_attr_value(element, attr_name):
    for k, v in element.attrib.items():
        if _strip_ns(k) == attr_name:
            return v
    return None

def _extrair_valor(root, path):
    parts, attr_name = _parse_path(path)
    if parts and _strip_ns(root.tag) == parts[0]:
        parts = parts[1:]
    nodes = [root]
    for part in parts:
        next_nodes = []
        for node in nodes:
            for child in node:
                if _strip_ns(child.tag) == part:
                    next_nodes.append(child)
        nodes = next_nodes
        if not nodes:
            return None
    if attr_name:
        values = [_get_attr_value(n, attr_name) for n in nodes]
        values = [v for v in values if v is not None]
    else:
        # Captura o texto e garante que não haja problemas de encoding
        values = []
        for n in nodes:
            txt = (n.text or "").strip()
            if txt:
                try:
                    # Tenta converter possíveis resquícios de encoding errado
                    # Isso resolve casos onde o XML foi lido como latin-1 mas é utf-8
                    txt = txt.encode('latin-1').decode('utf-8')
                except (UnicodeEncodeError, UnicodeDecodeError):
                    pass # Se falhar, mantém o original (provavelmente já está em utf-8)
                values.append(txt)

    values = list(dict.fromkeys(values))
    return values[0] if values else None

# ── SKILLS ───────────────────────────────────────────────────

def skill_listar_campos(xml_path: str) -> dict:
    """
    Lista todos os campos extraíveis do XML.
    Retorna dict {str(indice): caminho_completo}.
    """
    root = _load_xml(xml_path)
    fields = sorted(_collect_extractable_fields(root))
    return {str(i): campo for i, campo in enumerate(fields, start=1)}


def skill_mapear_campos(campos_desejados: list, catalogo: dict) -> dict:
    """
    Recebe a lista de campos canônicos desejados e o catálogo do XML.
    Retorna o mapeamento:
        {
            "nome_canonico": {
                "indice": int,
                "path": str,
                "confianca": "alta" | "media" | "baixa"
            },
            ...
        }
    Esta função é chamada PELO AGENTE via tool calling.
    O LLM preenche o mapeamento com base no catálogo.
    """
    # Esta skill é resolvida pelo LLM — o retorno vem do tool call
    pass


def skill_extrair_xml(xml_path: str, mapeamento: dict) -> dict:
    """
    Extrai os campos de um único XML usando o mapeamento canônico.
    Campos com confiança 'baixa' recebem asterisco no valor.
    Retorna dict {nome_canonico: valor}.
    """
    root = _load_xml(xml_path)
    registro = {"arquivo": os.path.basename(xml_path)}

    for campo_canonico, info in mapeamento.items():
        path      = info.get("path")
        confianca = info.get("confianca", "alta")

        if not path:
            registro[campo_canonico] = None
            continue

        valor = _extrair_valor(root, path)

        if valor is not None and confianca == "baixa":
            valor = f"{valor} *"

        registro[campo_canonico] = valor

    return registro

# ── SIMILARIDADE DE ESQUEMAS ─────────────────────────────────

def _fingerprint(catalogo: dict) -> str:
    """Hash estável do conjunto de campos do catálogo."""
    campos = tuple(sorted(catalogo.values()))
    return hashlib.md5(json.dumps(campos).encode()).hexdigest()

def _similaridade_jaccard(set_a: set, set_b: set) -> float:
    """Similaridade de Jaccard entre dois conjuntos de campos."""
    if not set_a and not set_b:
        return 1.0
    intersecao = len(set_a & set_b)
    uniao      = len(set_a | set_b)
    return intersecao / uniao if uniao > 0 else 0.0

def _encontrar_schema_compativel(
    catalogo_novo: dict,
    schema_cache: dict,
    limiar: float = LIMIAR_SIMILARIDADE
) -> str | None:
    """
    Verifica se o catálogo novo é suficientemente similar
    a algum esquema já mapeado.
    Retorna o schema_id compatível ou None.
    """
    campos_novos = set(catalogo_novo.values())
    melhor_id    = None
    melhor_sim   = 0.0

    for schema_id, schema in schema_cache.items():
        campos_schema = set(schema["catalogo"].values())
        sim = _similaridade_jaccard(campos_novos, campos_schema)
        if sim >= limiar and sim > melhor_sim:
            melhor_sim = sim
            melhor_id  = schema_id

    return melhor_id

# ── DEFINIÇÃO DAS TOOLS PARA O GROQ ──────────────────────────

def _build_tools(campos_desejados: list) -> list:
    campos_str = ", ".join(f'"{c}"' for c in campos_desejados)
    return [
        {
            "type": "function",
            "function": {
                "name": "skill_mapear_campos",
                "description": (
                    "Analisa o catálogo de campos de um XML e mapeia cada campo "
                    "canônico desejado para o caminho correspondente no XML. "
                    "Para cada campo canônico, escolha o caminho mais semanticamente "
                    "próximo do catálogo. Indique a confiança: "
                    "'alta' = correspondência clara, "
                    "'media' = provável mas não óbvio, "
                    "'baixa' = melhor candidato disponível mas incerto. "
                    f"Os campos canônicos a mapear são: [{campos_str}]."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "mapeamento": {
                            "type": "object",
                            "description": (
                                "Dicionário onde cada chave é um nome canônico "
                                "e o valor é um objeto com 'indice' (int), "
                                "'path' (string do caminho completo) e "
                                "'confianca' ('alta', 'media' ou 'baixa'). "
                                "Se não houver nenhum candidato, use path=null."
                            ),
                            "additionalProperties": {
                                "type": "object",
                                "properties": {
                                    "indice":    {"type": "integer"},
                                    "path":      {"type": ["string", "null"]},
                                    "confianca": {
                                        "type": "string",
                                        "enum": ["alta", "media", "baixa"]
                                    }
                                },
                                "required": ["indice", "path", "confianca"]
                            }
                        }
                    },
                    "required": ["mapeamento"]
                }
            }
        }
    ]

# ── AGENTE ───────────────────────────────────────────────────

class AgenteXML:
    def __init__(self, api_key: str, modelo: str = MODELO):
        self.client       = Groq(api_key=api_key)
        self.modelo       = modelo
        self.schema_cache = {}   # {schema_id: {fingerprint, catalogo, mapeamento, arquivo_modelo}}
        self._schema_seq  = 0

    # ── mapeamento via LLM ──────────────────────────────────

    def _mapear_com_llm(
        self,
        campos_desejados: list,
        catalogo: dict,
        arquivo_modelo: str,
        verbose: bool
    ) -> dict:
        """
        Chama o LLM para mapear campos_desejados → caminhos do catálogo.
        Retorna o mapeamento canônico.
        """
        tools = _build_tools(campos_desejados)

        catalogo_formatado = "\n".join(
            f"[{idx}] {path}" for idx, path in catalogo.items()
        )

        campos_str = "\n".join(f"- {c}" for c in campos_desejados)

        messages = [
            {
                "role": "system",
                "content": (
                    "Você é um especialista em XML de notas fiscais eletrônicas brasileiras "
                    "(NFS-e, NF-e, CT-e). Sua tarefa é mapear campos semânticos para "
                    "caminhos exatos de um catálogo XML. Seja preciso e use a tool "
                    "skill_mapear_campos para retornar o mapeamento."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Arquivo modelo: {arquivo_modelo}\n\n"
                    f"Campos canônicos que preciso mapear:\n{campos_str}\n\n"
                    f"Catálogo completo do XML ({len(catalogo)} campos):\n"
                    f"{catalogo_formatado}\n\n"
                    "Use a tool skill_mapear_campos para retornar o mapeamento completo."
                )
            }
        ]

        response = self.client.chat.completions.create(
            model=self.modelo,
            messages=messages,
            tools=tools,
            tool_choice={"type": "function", "function": {"name": "skill_mapear_campos"}},
            temperature=0
        )

        msg = response.choices[0].message

        if not msg.tool_calls:
            raise RuntimeError("LLM não retornou tool call para skill_mapear_campos.")

        args = json.loads(msg.tool_calls[0].function.arguments)
        mapeamento = args.get("mapeamento", {})

        if verbose:
            alta   = sum(1 for v in mapeamento.values() if v.get("confianca") == "alta")
            media  = sum(1 for v in mapeamento.values() if v.get("confianca") == "media")
            baixa  = sum(1 for v in mapeamento.values() if v.get("confianca") == "baixa")
            nulos  = sum(1 for v in mapeamento.values() if not v.get("path"))
            print(f"   📊 Mapeamento: {alta} alta | {media} média | {baixa} baixa | {nulos} não encontrado")

        return mapeamento

    # ── registro de schema ──────────────────────────────────

    def _registrar_schema(
        self,
        catalogo: dict,
        mapeamento: dict,
        arquivo_modelo: str
    ) -> str:
        self._schema_seq += 1
        schema_id = f"schema_{self._schema_seq:03d}"
        self.schema_cache[schema_id] = {
            "fingerprint":    _fingerprint(catalogo),
            "catalogo":       catalogo,
            "mapeamento":     mapeamento,
            "arquivo_modelo": arquivo_modelo,
            "campos_set":     set(catalogo.values())
        }
        return schema_id

    # ── processamento principal ─────────────────────────────

    def processar(
        self,
        pasta_xml: str,
        campos_desejados: list,
        recursive: bool = False,
        verbose: bool = True
    ) -> pd.DataFrame:
        """
        Processa todos os XMLs de uma pasta, detectando automaticamente
        múltiplos esquemas e remapeando quando necessário.

        Parâmetros:
            pasta_xml       : pasta com os arquivos XML
            campos_desejados: lista de nomes canônicos desejados
                              ex: ["numero_nota", "data_emissao", "cnpj_tomador"]
            recursive       : busca em subpastas
            verbose         : imprime progresso

        Retorna:
            pd.DataFrame com colunas = campos_desejados + "arquivo" + "schema_id"
        """
        pattern = (
            os.path.join(pasta_xml, "**", "*.xml") if recursive
            else os.path.join(pasta_xml, "*.xml")
        )
        xml_files = sorted(glob.glob(pattern, recursive=recursive))

        if not xml_files:
            raise FileNotFoundError(f"Nenhum XML encontrado em: {pasta_xml}")

        if verbose:
            print(f"🤖 Agente XML Multi-Schema | Modelo: {self.modelo}")
            print(f"📁 {len(xml_files)} arquivo(s) encontrado(s) em: {pasta_xml}")
            print(f"📋 {len(campos_desejados)} campo(s) canônico(s) solicitado(s)")
            print("─" * 65)

        registros        = []
        erros            = []
        schemas_usados   = {}   # arquivo → schema_id

        for i, xml_file in enumerate(xml_files, start=1):
            nome = os.path.basename(xml_file)

            if verbose:
                print(f"\n[{i:03d}/{len(xml_files):03d}] {nome}")

            try:
                # 1. Gera catálogo do arquivo atual
                catalogo_atual = skill_listar_campos(xml_file)

                # 2. Verifica se já existe schema compatível
                schema_id = _encontrar_schema_compativel(
                    catalogo_atual, self.schema_cache
                )

                if schema_id:
                    if verbose:
                        sim = _similaridade_jaccard(
                            set(catalogo_atual.values()),
                            self.schema_cache[schema_id]["campos_set"]
                        )
                        print(f"   ♻️  Schema reutilizado: {schema_id} "
                              f"(similaridade: {sim:.1%})")
                    mapeamento = self.schema_cache[schema_id]["mapeamento"]

                else:
                    # 3. Novo schema — chama LLM para mapear
                    if verbose:
                        print(f"   🆕 Novo schema detectado — chamando LLM para mapear...")

                    mapeamento = self._mapear_com_llm(
                        campos_desejados=campos_desejados,
                        catalogo=catalogo_atual,
                        arquivo_modelo=xml_file,
                        verbose=verbose
                    )

                    schema_id = self._registrar_schema(
                        catalogo=catalogo_atual,
                        mapeamento=mapeamento,
                        arquivo_modelo=xml_file
                    )

                    if verbose:
                        print(f"   ✅ Novo schema registrado: {schema_id}")

                # 4. Extrai os dados do arquivo atual
                registro = skill_extrair_xml(xml_file, mapeamento)
                registro["schema_id"] = schema_id
                registros.append(registro)
                schemas_usados[nome] = schema_id

            except Exception as e:
                erros.append({"arquivo": nome, "erro": str(e)})
                if verbose:
                    print(f"   ❌ Erro: {e}")

        # ── monta DataFrame ──────────────────────────────────
        df = pd.DataFrame(registros)

        # Garante que todas as colunas canônicas existam
        for campo in campos_desejados:
            if campo not in df.columns:
                df[campo] = None

        # Ordena colunas: arquivo, schema_id, campos canônicos
        colunas_fixas   = ["arquivo", "schema_id"]
        colunas_campos  = [c for c in campos_desejados if c in df.columns]
        colunas_extras  = [c for c in df.columns if c not in colunas_fixas + colunas_campos]
        df = df[colunas_fixas + colunas_campos + colunas_extras]

        # ── relatório final ──────────────────────────────────
        if verbose:
            print("\n" + "═" * 65)
            print("📊 RELATÓRIO FINAL")
            print("═" * 65)
            print(f"✅ Arquivos processados : {len(registros)}")
            print(f"❌ Erros                : {len(erros)}")
            print(f"🗂️  Schemas detectados  : {len(self.schema_cache)}")

            for sid, schema in self.schema_cache.items():
                qtd = sum(1 for v in schemas_usados.values() if v == sid)
                print(f"   • {sid}: {qtd} arquivo(s) — modelo: {os.path.basename(schema['arquivo_modelo'])}")

            # Campos com confiança baixa
            campos_duvidosos = []
            for sid, schema in self.schema_cache.items():
                for campo, info in schema["mapeamento"].items():
                    if info.get("confianca") == "baixa":
                        campos_duvidosos.append((sid, campo, info.get("path", "?")))

            if campos_duvidosos:
                print(f"\n⚠️  Campos com confiança BAIXA (marcados com *):")
                for sid, campo, path in campos_duvidosos:
                    print(f"   [{sid}] {campo} → {path}")

            if erros:
                print(f"\n❌ Arquivos com erro:")
                for e in erros:
                    print(f"   {e['arquivo']}: {e['erro']}")

            print("═" * 65)
            print(f"DataFrame: {len(df)} linha(s) × {len(df.columns)} coluna(s)")

        return df

    def relatorio_mapeamento(self) -> pd.DataFrame:
        """
        Retorna um DataFrame com o mapeamento completo de todos os schemas.
        Útil para auditoria e validação.
        """
        rows = []
        for schema_id, schema in self.schema_cache.items():
            for campo_canonico, info in schema["mapeamento"].items():
                rows.append({
                    "schema_id":      schema_id,
                    "arquivo_modelo": os.path.basename(schema["arquivo_modelo"]),
                    "campo_canonico": campo_canonico,
                    "path_xml":       info.get("path"),
                    "confianca":      info.get("confianca"),
                    "indice":         info.get("indice")
                })
        return pd.DataFrame(rows)

#RUN

In [ ]:
#── USO ──────────────────────────────────────────────────────

agente = AgenteXML(api_key=GROQ_API_KEY, modelo=MODELO)

df = agente.processar(
    pasta_xml="/content/drive/MyDrive/tmp/XMLs Extracao/",
    campos_desejados=[
        "numero_nota",
        "data_emissao",
        "competencia",
        "cnpj_prestador",
        "razao_social_prestador",
        "cnpj_tomador",
        "razao_social_tomador",
        "municipio_incidencia",
        "valor_total",
        "base_calculo",
        "aliquota",
        "valor_iss",
        "codigo_servico",
        "discriminacao_servico",
    ],
    verbose=True
)

# Visualiza resultado
print(df.head())

# Exporta para Excel
df.to_excel("notas_fiscais.xlsx", index=False, engine='openpyxl')
print("✅ Exportado para notas_fiscais.xlsx")

# Auditoria do mapeamento
df_mapeamento = agente.relatorio_mapeamento()
print(df_mapeamento)